<img src="images/banner.png" style="width: 100%;">

In [1]:
import numpy as np
from numpy.testing import assert_array_equal, assert_almost_equal, assert_array_almost_equal

# Backpropagation in RNNs

In this notebook, we'll show how to perform BPTT through the worked example discussed in class.

Recall the example:

Consider the problem of predicting the next bit in a binary sequence:

<img src="images/signal.png" style="width: 30%;">

Construct an RNN that predicts the next value using the previous 2 values of the sequence with a learning rate of 0.2.

<img src="images/rnn.png" style="width: 80%;">

## 1 Forward Pass

In [2]:
import numpy as np


def sigmoid(x):
    """Return the sigmoid of x"""
    return 1 / (1 + np.exp(-x))


def sech(x):
    """Return the hyperbolic secant of x"""
    return 1 / np.cosh(x)

In [3]:
# Initial conditions
V_0 = np.array([1])
U_0 = np.array([0.5])
W_0 = np.array([0.5])

# Forward pass
def forward(X, U, W, V, h_0=None):
    """Perform the forward pass of the simple RNN architecture"""
    if h_0 is None:
        h_0 = np.array([0])

    # Initialize h and state vector H
    H = [h_0]
    h = h_0
    for x in X:
        h = np.tanh(W*h + U*x)
        H.append(h)

    return sigmoid(V*h), H

In [4]:
X = np.array([1, 0])
forward(X, U_0, W_0, V_0)

(array([0.55651561]), [array([0]), array([0.46211716]), array([0.22703261])])

In [5]:
# Test function according to our computation by hand
assert forward(X, U_0, W_0, V_0)[0] == sigmoid(np.tanh(0.5 * np.tanh(0.5)))

## 2 BPTT

Given the **Sum of the Squared Error (SSE)** as our loss function, defined as:

$$
L = \frac{1}{2}(y - o)^2
$$

We've derived that the gradient of the loss function wrt the weight parameters are:

<img src="images/grad_loss.png" style="width: 60%;">

In [6]:
def loss(y, o):
    """Return the squared error loss"""
    return 0.5*(y - o)**2


def grad_output(y, o, V, H):
    """Return the gradient of the loss function wrt output"""
    return -(y - o)*sigmoid(V*H[2])*(1 - sigmoid(V*H[2]))


def grad_V(y, o, V, H):
    """Return the gradient of the loss wrt V"""
    return grad_output(y, o, V, H)*H[2]


def grad_h2(y, o, V, H):
    """Return the gradient of the loss wrt h2"""
    return grad_output(y, o, V, H)*V


def grad_U(y, o, X, U, W, V, H):
    """Return the gradient of the loss wrt U"""
    dh1_dU = (sech(W*H[0] + U*X[0])**2) * X[0]
    dh2_dU = (sech(W*H[1] + U*X[1])**2) * (X[1] + W*dh1_dU)

    return grad_h2(y, o, V, H) * dh2_dU

def grad_W(y, o, X, U, W, V, H):
    """Return the gradient of the loss wrt W"""
    dh1_dW = (sech(W*H[0] + U*X[0])**2) * H[0]
    dh2_dW = (sech(W*H[1] + U*X[1])**2) * (H[1] + W*dh1_dW)

    return grad_h2(y, o, V, H) * dh2_dW

In [7]:
# Sample Data point
X = np.array([1, 0])
y = np.array([1])

# Initial conditions
V_0 = np.array([1])
U_0 = np.array([0.5])
W_0 = np.array([0.5])

o, H = forward(X, U_0, W_0, V_0)

print(f"Loss: {loss(y, o)[0]}")

Loss: 0.09833920297158642


In [8]:
h2 = np.tanh(0.5 * np.tanh(0.5))
assert_almost_equal(grad_V(y, o, V_0, H), -(((1 - sigmoid(h2)))**2) * sigmoid(h2) * h2)

In [9]:
dL_dh2 = -(((1 - sigmoid(h2)))**2) * sigmoid(h2)
assert_almost_equal(grad_h2(y, o, V_0, H), dL_dh2)

In [10]:
assert_almost_equal(grad_U(y, o, X, U_0, W_0, V_0, H), dL_dh2 * (sech(0.5*np.tanh(0.5))**2) * (0.5 * (sech(0.5)**2)))

In [11]:
assert_almost_equal(grad_W(y, o, X, U_0, W_0, V_0, H), dL_dh2 * (sech(0.5*np.tanh(0.5))**2) * (np.tanh(0.5)))

## 3 Gradient Descent Algorithm

Now that we have functions that allows us to compute the loss function and gradient of the weights in terms of the output and input matrices, we can perform the gradient descent algorithm and update our network weights accordingly.

$$
V^{t+1} = V^{t} - 0.2 \cdot \frac{dL}{dV^t}
$$
$$
U^{t+1} = U^{t} - 0.2 \cdot \frac{dL}{dU^t}
$$
$$
W^{t+1} = W^{t} - 0.2 \cdot \frac{dL}{dW^t}
$$

### First Iteration

In [12]:
# Learning rate
alpha = 0.2

# Initial conditions
U = U_0
W = W_0
V = V_0

V = V - alpha*grad_V(y, o, V, H)
U = U - alpha*grad_U(y, o, X, U, W, V, H)
W = W - alpha*grad_W(y, o, X, U, W, V, H)

In [13]:
print(f"Loss: {loss(y, forward(X, U, W, V)[0])[0]}")

Loss: 0.09741320131405544


### Iteration Until Convergence

Perform the gradient descent successively until the SSE reaches below or equal to `1e-6`. Place the final weight matrix in the variable named `w_final`, and the number of iterations needed to convergence as `num_iter`.

In [14]:
# Tolerance setting
tol = 1e-5
alpha = 0.2

# Initial values
U = U_0
W = W_0
V = V_0
o, H = forward(X, U, W, V)

# Gradient Descent Algorithm
num_iter = 0
while loss(y, o) >= tol:
    o, H = forward(X, U, W, V)
    V = V - alpha*grad_V(y, o, V, H)
    U = U - alpha*grad_U(y, o, X, U, W, V, H)
    W = W - alpha*grad_W(y, o, X, U, W, V, H)
    num_iter += 1

In [15]:
print(f"It took {num_iter:,} iterations to converge, with final loss value of {loss(y, o)}")

It took 127,077 iterations to converge, with final loss value of [9.99993555e-06]


In [16]:
forward(X, U, W, V)

(array([0.9955279]), [array([0]), array([0.93072407]), array([0.98247007])])

<img src="images/banner-down.png" style="width: 100%;">